In [103]:
# ============================================
# scRNA-seq basic Scanpy workflow
# ============================================
#
# This notebook:
# 1. Loads 10x matrix.mtx.gz datasets
# 2. Merges 6 mtxitions
# 3. Performs preprocessing
# 4. Computes PCA + UMAP
# 5. Plots UMAP colored by mtxition
#
# Required directory structure:
#
# ============================================
# 1. Install packages (run once)
# ============================================

%pip install scanpy anndata matplotlib leidenalg

In [104]:
# ============================================
# DONT CHANGE:Download data from GEO
# ============================================ 
!rm -rf data
!mkdir data
!wget "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE196280&format=file" -O GSE196280.tar
!tar -xvf GSE196280.tar --directory data/

In [105]:

import scanpy as sc
import re
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path


In [106]:
# ============================================
# DONT CHANGE: Load the matrix files
# ============================================ 
# 
# # Plot settings
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(6, 6))

base = Path("data/")
adatas = []
#search all matrix.mtx.gz files in data/ and remove the matrix.mtx.gz suffix to get the mtxition name
for mtx in base.iterdir():
    if not mtx.is_file() or not mtx.name.endswith("matrix.mtx.gz"):
        continue

    mtxition = mtx.name.replace(".matrix.mtx.gz", "")

    #create a subdir called mtxition and move all files with the same mtxition name into it
    dir = base / mtxition
    dir.mkdir(exist_ok=True)
    for file in base.iterdir():
        if file.is_file() and file.name.startswith(mtxition):
            file.rename(dir / file.name.replace(mtxition+".", ""))


for dir in base.iterdir():
    if not dir.is_dir():
        continue

    condition = re.sub(r"GSM[^_]+_", "", dir.name)  
    print(f"Loading {condition} from {dir}...")

    adata = sc.read_10x_mtx(
        dir,
        var_names="gene_symbols",
        cache=True
    )

    # Add metadata
    adata.obs["name"] = condition

    # Make unique cell IDs
    adata.obs_names_make_unique()

    # Optional: prefix barcodes with mtxition
    adata.obs_names = [
        f"{condition}_{cell}"
        for cell in adata.obs_names
    ]

    adatas.append(adata)

    adata = ad.concat(
    adatas,
    axis=0,
    join="outer",
    label="name",
    keys=[adata.obs["name"][0] for adata in adatas],
    index_unique="-"
)


In [108]:
# ============================================
# 6. Basic QC metrics
# ============================================

# TODO: Follow the tutotial "Quality Control" to evaluate mitochondrial and ribosomal gene contamination
# https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#quality-control

# View summary
adata.obs.head()


In [109]:
# ============================================
# 7. QC plots
# ============================================

# TODO: Make the violin plot for QC metrics
# https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#quality-control



In [110]:

# TODO: And the scatter plot
# https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#quality-control



In [112]:

# ============================================
# 8. Filter low-quality cells/genes
# ============================================

# Identify dublets
sc.pp.scrublet(adata, batch_key="sample")

# Filter cells
sc.pp.filter_cells(adata, min_genes=200)

# Filter genes
sc.pp.filter_genes(adata, min_cells=3)

# Remove high mitochondrial cells
adata = adata[adata.obs.pct_counts_mt < 15].copy()

print(adata)


In [113]:
# ============================================
# 9. Normalize and log transform
# ============================================

# Save raw counts
adata.layers["counts"] = adata.X.copy()

# Normalize counts per cell
sc.pp.normalize_total(
    adata,
    target_sum=1e4
)

# Log transform
sc.pp.log1p(adata)


In [114]:
# ============================================
# 10. Highly variable genes
# ============================================

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=False, #want to keep all the genes for later visualization
    flavor="seurat"
)

# Plot HVGs
sc.pl.highly_variable_genes(adata)

In [115]:
# ============================================
# 11. Scale data
# ============================================

sc.pp.scale(
    adata,
    max_value=10
)

In [116]:
# ============================================
# 12. PCA
# ============================================

# TODO: Generate PCA and variance ratio plot as described in 
# https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#dimensionality-reduction



In [117]:
# ============================================
# Plot PCA
# ============================================
# 
sc.pl.pca(
    adata,
    color="name",
    components=["1,2"]
)

# ============================================
# TODO: Also Plot PC2 vs PC3, PC3 vs PC4
# https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#dimensionality-reduction


sc.pl.pca(

)

In [118]:
# ============================================
# 13. Compute neighbors
# ============================================
#https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#nearest-neighbor-graph-construction-and-visualization

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)


# ============================================
# 14. UMAP
# ============================================

sc.tl.umap(adata)


In [119]:
# ============================================
# 15. Leiden clustering
# ============================================

sc.tl.leiden(
    adata,
    resolution=0.5
)

In [120]:
# ============================================
# 16. Plot UMAPs
# ============================================

# Colored by name
sc.pl.umap(
    adata,
    color="name",
    frameon=False
)

# TODO: color by Leiden cluster
#
sc.pl.umap(
    adata,
    color=,
    legend_loc="on data",
    frameon=False
)

In [127]:
# ============================================
# 17. Marker gene visualization
# ============================================


marker_genes = [
    "SOX2",  
    "NANOG",
    "GATA3", 
    "CLDN4",
    "TP63",
    "CGB",
    "TMEM88", 
    "KRT18",  
    "TBXT" 
    "SOX9",
    "TBX3",
    "CDX1",
    "CDX2",
    "TFAP2A",
    "TFAP2C", 
    "CD24", 
]

existing_markers = [
    g for g in marker_genes
    if g in adata.var_names
]

print(existing_markers)

# plot panel with 3 plots per row
sc.pl.umap(
    adata,
    color=existing_markers,
    ncols=3
)

In [128]:
# ============================================
# 18. Find marker genes per cluster
# ============================================

sc.tl.rank_genes_groups(
    adata,
    groupby="leiden",
    method="wilcoxon"
)

# Show top markers
sc.pl.rank_genes_groups(
    adata,
    n_genes=20,
    sharey=False
)

In [ ]:

# ============================================
# 19. Save processed object
# ============================================

adata.write("combined_scRNAseq.h5ad")

print("\nSaved combined_scRNAseq.h5ad")


# ============================================
# 20. Reload later
# ============================================

# adata = sc.read_h5ad("combined_scRNAseq.h5ad")